In [43]:
from pwn import *
import hlextend
import base64

In [ ]:
SECRET_LENGTH = 32

In [44]:
session = remote('45fbe9d0de3e101690804d96-1024-intro-crypto-2.challenge.cscg.live', 1337, ssl=True)

[x] Opening connection to 45fbe9d0de3e101690804d96-1024-intro-crypto-2.challenge.cscg.live on port 1337
[x] Opening connection to 45fbe9d0de3e101690804d96-1024-intro-crypto-2.challenge.cscg.live on port 1337: Trying 147.75.204.231
[+] Opening connection to 45fbe9d0de3e101690804d96-1024-intro-crypto-2.challenge.cscg.live on port 1337: Done


In [45]:
session.recvuntil(b'Enter your choice: ')
session.sendline(b'1')
session.recvuntil(b'What is you name? ')
session.sendline(b'test')
session.recvuntil(b'What is your favorite animal? ')
session.sendline(b'cat')
session.recvuntil(b'Here is your access token:')
secure_token_b64 = session.recvline(keepends=False)
secure_token_b64

b' bmFtZT10ZXN0fGFuaW1hbD1jYXR8YWRtaW49ZmFsc2V8bWFjPTUxNTBlZTAzZTMyOTA3NTY1MjM0MGU0MmU3NzJkYTVkN2U3ODhmYjY='

In [46]:
secure_token = base64.b64decode(secure_token_b64).decode()
token, mac = secure_token.split('|mac=')
secure_token, token, mac

('name=test|animal=cat|admin=false|mac=5150ee03e329075652340e42e772da5d7e788fb6',
 'name=test|animal=cat|admin=false',
 '5150ee03e329075652340e42e772da5d7e788fb6')

In [ ]:
extender = hlextend.new('sha1')
extended_token = extender.extend(b'|admin=true', token.encode(), SECRET_LENGTH, mac)
extended_mac = extender.hexdigest()
new_secure_token = extended_token + b'|mac=' + extended_mac.encode()
new_secure_token_b64 = base64.b64encode(new_secure_token).decode()
extended_mac, new_secure_token, new_secure_token_b64

('3a47e9da727b1335c697e741037a2fc35bdc19ac',
 b'name=test|animal=cat|admin=false\x80\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x02\x00|admin=true|mac=3a47e9da727b1335c697e741037a2fc35bdc19ac',
 'bmFtZT10ZXN0fGFuaW1hbD1jYXR8YWRtaW49ZmFsc2WAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIAfGFkbWluPXRydWV8bWFjPTNhNDdlOWRhNzI3YjEzMzVjNjk3ZTc0MTAzN2EyZmMzNWJkYzE5YWM=')

In [48]:
session.recvuntil(b'Enter your choice: ')
session.sendline(b'3')
session.recvuntil(b'Enter access token: ')
session.sendline(new_secure_token_b64.encode())

print(session.recvline(keepends=False).decode())

The flag is CSCG{sh0uld_have_us3d_HMAC_or_KMAC_instead!}


In [49]:
session.close()

[*] Closed connection to 45fbe9d0de3e101690804d96-1024-intro-crypto-2.challenge.cscg.live port 1337
